# Load Q K V

In [1]:
import torch
import math

# ——— Load and prepare q0, k0 as PyTorch tensors ———
q_full = torch.load("subset_qk/block_1_q_proj_batch_6.pt", map_location="cpu")
k_full = torch.load("subset_qk/block_1_k_proj_batch_6.pt", map_location="cpu")

q = q_full[0]  # shape [L, d_model]
k = k_full[0]
v = torch.zeros_like(q) 

L, d_model = q.shape
num_heads  = 32
d_head     = d_model // num_heads
# pick head 15 and first `sample` positions
sample = 4096
def sampling(q, k, sample, POS, normalize=False):
    q0 = (
        q
        .view(L, num_heads, d_head)
        .permute(1, 0, 2)[POS, :sample]
    )   # shape [sample, d_head]
    k0 = (
        k
        .view(L, num_heads, d_head)
        .permute(1, 0, 2)[POS, :sample]
    )

    q0 = q0 / 128 ** 0.25
    k0 = k0 / 128 ** 0.25
    if normalize:
        q0 = unit_norm_normalize(q0)
        k0 = unit_norm_normalize(k0)
    
    v0 = (q0 + 3 * k0)/ 4
    return q0, k0, v0
def unit_norm_normalize(matrix):
    """
    对矩阵进行单位范数归一化（L2归一化）
    matrix: (N, M) 输入矩阵
    returns: (N, M) 归一化后的矩阵，每行的L2范数为1
    """
    # 计算每行的L2范数
    row_norms = torch.norm(matrix, dim=1, keepdim=True)
    
    # 归一化
    normalized_matrix = matrix / row_norms
    
    return normalized_matrix





# Usage

# Utilities for True Softmax

In [2]:
# get real result
def true_softmax(q0, k0):
    dot = q0 @ k0.T
    true_val = torch.exp(dot - dot.max(dim=1, keepdim=True)[0])
    true_val /= true_val.sum(dim=1, keepdim=True)
    return true_val

# Utilities for RF attention

In [3]:


def rfa_q_feature_mapping_blocks(q0, P=8, D=2000):
    """
    保持输入为P个块，每个块D个向量，输出展平为P*D维
    q0: (N, d) 查询数组
    P: 块数 (默认为8)
    D: 每个块的向量数 (默认为2000)
    returns: (N, P*D) 展平后的特征
    """
    # 1) 预处理和缩放
    X = q0.to(torch.float16)   # (N, d)
    N, d = X.shape

    # 2) 为每个块生成随机权重
    w_blocks = torch.sign(torch.randn(P, D, d, device=X.device)).to(torch.float16)  # (P, D, d)

    # 3) 计算每个块的投影
    proj_blocks = torch.zeros((N, P, D), device=X.device, dtype=torch.float16)  # (N, P, D)
    for p in range(P):
        proj_blocks[:, p, :] = X @ w_blocks[p].T  # (N, D)

    # 4) 计算每个块的归一化因子
    facts = torch.tensor([math.sqrt(math.factorial(p+1)) for p in range(P)],
                     dtype=torch.float16, device=X.device)     # (P,)
    normalizer = torch.sqrt(torch.tensor(D, dtype=torch.float16)) * facts
    normalizer = normalizer.view(1, P, 1)  # (1, P, 1)

    # 5) 应用归一化
    phi_blocks = proj_blocks / normalizer  # (N, P, D)

    # 6) 展平为(N, P*D)
    phi_flat = phi_blocks.reshape(N, P*D)  # (N, P*D)
    return phi_flat

In [4]:
def approx(q0, k0, v0, P, D):
    approx_result = torch.zeros_like(q0)
    # get first row of result
    phi_q0 = rfa_q_feature_mapping_blocks(q0, P, D) # Shape 4096, P* D
    # result is 
    phi_k0 = rfa_q_feature_mapping_blocks(k0, P, D) # Shape 4096, P* D
    # Sha
    phi_q0 += 1/math.sqrt(P*D) #麦克劳林展开的1，分摊在每个dim里
    phi_k0 += 1/math.sqrt(P*D)

    # 向量化计算rest矩阵
    # phi_k0: (4096, P*D), v0: (4096, 128)
    # 使用矩阵乘法直接计算
    rest = torch.matmul(phi_k0.t(), v0)  # Shape (P*D, 128)

    # Get top of RF attention
    top = torch.matmul(phi_q0, rest)  # Shape (4096, 128)

    # Get Bottom of softmax kernel version random feature
    row_sum = phi_k0.sum(dim=0)  # Shape (P*D,)
    # 向量化计算bottom
    bottom = torch.matmul(phi_q0, row_sum)  # Shape (4096,)
    # 广播除法
    approx_result = top / bottom.unsqueeze(1)  # Shape (4096, 128)
    return approx_result

In [5]:
def report_error(record_approx_values, true_val):
    return torch.norm(record_approx_values - true_val) / torch.norm(true_val)

# Next is to test normalized QKV

In [ ]:
sample=4096
real_result_full = torch.zeros((4096, 4096))
for i in range(num_heads):
    q0, k0, v0= sampling(q, k, sample, i, True)
    real_result = true_softmax(q0, k0)
    real_result = real_result @ v0
    real_result_full[:, i * 128: i*128+128] = real_result
    
P, D, d = 2, 800, 128
sample=4096
approx_result_full = torch.zeros((4096, 4096))
for i in range(num_heads):
    q0, k0, v0= sampling(q, k, sample, i, True)
    approx_result = approx(q0, k0, v0, P, D)
    approx_result_full[:, i * 128: i*128+128] = approx_result




In [ ]:
report_error(real_result_full, approx_result_full)

# To be done Revise MHA calling error

In [9]:
import torch
import math
import torch.nn as nn
# ——— Load and prepare q0, k0 as PyTorch tensors ———
q_full = torch.load("subset_qk/block_1_q_proj_batch_6.pt", map_location="cpu")
k_full = torch.load("subset_qk/block_1_k_proj_batch_6.pt", map_location="cpu")

q = q_full[0]  # shape [L, d_model]
k = k_full[0]

L, d_model = q.shape
num_heads  = 32
d_head     = d_model // num_heads

v = (q + 3 * k) / 4
# 确保输入是float32类型
q = q.to(torch.float32)
k = k.to(torch.float32)
v = v.to(torch.float32)

# 确保输入维度正确：(seq_len, batch_size, embed_dim)
q_in = q.unsqueeze(1)  # [seq_len, 1, d_model]
k_in = k.unsqueeze(1)  # [seq_len, 1, d_model]
v_in = v.unsqueeze(1)  # [seq_len, 1, d_model]



# 创建MultiheadAttention实例
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads)

# 计算注意力
attention_result, attn_weights = mha(q_in, k_in, v_in)



# 移除批次维度
attention_result = attention_result.squeeze(1)  # [seq_len, d_model]9

import time
time.sleep(10)

KeyboardInterrupt: 

In [ ]:
report_error(real_result_full, attention_result)